# MetaPathPredict: Quick Start Guide

This notebook demonstrates the three training approaches available in MetaPathPredict:

1. **Configurable CNN** - Standard supervised learning with configurable kernel sizes
2. **Contrastive Learning** - Self-supervised pretraining with SimCLR/SupCon
3. **Deep Reinforcement Learning** - RL-based sequence classification

## Setup

In [ ]:
# Install dependencies if needed
# !pip install torch numpy h5py pydantic

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Preparation

First, let's create some synthetic data to demonstrate the training approaches.

In [ ]:
def generate_synthetic_data(num_samples: int = 1000, seq_length: int = 500, num_classes: int = 3):
    """
    Generate synthetic one-hot encoded sequences.
    
    Returns:
        X: Tensor of shape (num_samples, 4, seq_length)
        y: Tensor of shape (num_samples,) with class labels
    """
    # Generate random sequences (one-hot encoded)
    X = torch.zeros(num_samples, 4, seq_length)
    
    for i in range(num_samples):
        # Random nucleotide indices
        indices = torch.randint(0, 4, (seq_length,))
        X[i, indices, torch.arange(seq_length)] = 1.0
    
    # Generate labels
    y = torch.randint(0, num_classes, (num_samples,))
    
    return X, y

# Generate data
X_train, y_train = generate_synthetic_data(800)
X_val, y_val = generate_synthetic_data(200)

print(f"Training data: {X_train.shape}, {y_train.shape}")
print(f"Validation data: {X_val.shape}, {y_val.shape}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

## 2. Configurable CNN

The ConfigurableCNN allows you to choose different kernel size presets (small=5, medium=7, large=10) to capture different pattern scales.

In [ ]:
class ConfigurableCNN(nn.Module):
    """
    Configurable CNN for DNA sequence classification.
    
    Kernel presets:
    - small: kernel_size=5 (captures local motifs)
    - medium: kernel_size=7 (balanced)
    - large: kernel_size=10 (captures longer patterns)
    """
    
    KERNEL_PRESETS = {
        "small": 5,
        "medium": 7,
        "large": 10,
    }
    
    def __init__(
        self,
        input_channels: int = 4,
        num_classes: int = 3,
        kernel_preset: str = "medium",
        hidden_channels: list = None,
        dropout: float = 0.3,
    ):
        super().__init__()
        
        # Get kernel size from preset
        if kernel_preset not in self.KERNEL_PRESETS:
            raise ValueError(f"Unknown kernel preset: {kernel_preset}")
        kernel_size = self.KERNEL_PRESETS[kernel_preset]
        
        # Default hidden channels
        if hidden_channels is None:
            hidden_channels = [32, 64, 128]
        
        # Build convolutional layers
        layers = []
        in_channels = input_channels
        
        for out_channels in hidden_channels:
            layers.extend([
                nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Dropout(dropout),
            ])
            in_channels = out_channels
        
        self.conv_layers = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(hidden_channels[-1], num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_layers(x)
        x = self.global_pool(x).squeeze(-1)
        return self.classifier(x)
    
    def get_features(self, x: torch.Tensor) -> torch.Tensor:
        """Extract features before classification."""
        x = self.conv_layers(x)
        return self.global_pool(x).squeeze(-1)

In [ ]:
# Create models with different kernel presets
model_small = ConfigurableCNN(kernel_preset="small", num_classes=3)
model_medium = ConfigurableCNN(kernel_preset="medium", num_classes=3)
model_large = ConfigurableCNN(kernel_preset="large", num_classes=3)

print("Model parameters:")
print(f"  Small (k=5): {sum(p.numel() for p in model_small.parameters()):,}")
print(f"  Medium (k=7): {sum(p.numel() for p in model_medium.parameters()):,}")
print(f"  Large (k=10): {sum(p.numel() for p in model_large.parameters()):,}")

In [ ]:
def train_cnn(model, train_loader, val_loader, epochs=10, lr=1e-3):
    """
    Train CNN with standard supervised learning.
    """
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                
                _, predicted = outputs.max(1)
                total += y_batch.size(0)
                correct += predicted.eq(y_batch).sum().item()
        
        # Record metrics
        history["train_loss"].append(train_loss / len(train_loader))
        history["val_loss"].append(val_loss / len(val_loader))
        history["val_acc"].append(correct / total)
        
        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch+1}/{epochs} - "
                  f"Train Loss: {history['train_loss'][-1]:.4f}, "
                  f"Val Loss: {history['val_loss'][-1]:.4f}, "
                  f"Val Acc: {history['val_acc'][-1]:.4f}")
    
    return history

# Train the medium kernel model
print("Training CNN with medium kernel (k=7)...")
history = train_cnn(model_medium, train_loader, val_loader, epochs=10)

## 3. Contrastive Learning

Contrastive learning allows self-supervised pretraining on unlabeled sequences, learning useful representations before fine-tuning on labeled data.

In [ ]:
class SequenceAugmenter:
    """
    Augmentations for DNA sequences.
    """
    
    @staticmethod
    def random_crop(x: torch.Tensor, crop_ratio: float = 0.9) -> torch.Tensor:
        """Random crop along sequence dimension."""
        seq_len = x.shape[-1]
        crop_len = int(seq_len * crop_ratio)
        start = torch.randint(0, seq_len - crop_len + 1, (1,)).item()
        cropped = x[..., start:start + crop_len]
        # Pad back to original length
        return F.pad(cropped, (0, seq_len - crop_len))
    
    @staticmethod
    def random_mask(x: torch.Tensor, mask_ratio: float = 0.1) -> torch.Tensor:
        """Randomly mask positions."""
        mask = torch.rand(x.shape[-1]) > mask_ratio
        return x * mask.unsqueeze(0).expand_as(x)
    
    @staticmethod
    def gaussian_noise(x: torch.Tensor, std: float = 0.1) -> torch.Tensor:
        """Add Gaussian noise."""
        noise = torch.randn_like(x) * std
        return x + noise
    
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """Apply random augmentation."""
        aug_fn = np.random.choice([
            self.random_crop,
            self.random_mask,
            self.gaussian_noise,
        ])
        return aug_fn(x)

In [ ]:
class ProjectionHead(nn.Module):
    """MLP projection head for contrastive learning."""
    
    def __init__(self, input_dim: int, hidden_dim: int = 256, output_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), dim=-1)


class ContrastiveModel(nn.Module):
    """Contrastive learning model (SimCLR-style)."""
    
    def __init__(self, encoder: nn.Module, projection_dim: int = 128):
        super().__init__()
        self.encoder = encoder
        
        # Get encoder output dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 4, 500)
            encoder_dim = encoder.get_features(dummy).shape[-1]
        
        self.projection = ProjectionHead(encoder_dim, output_dim=projection_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.encoder.get_features(x)
        return self.projection(features)
    
    def get_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder.get_features(x)

In [ ]:
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.5) -> torch.Tensor:
    """
    NT-Xent loss (Normalized Temperature-scaled Cross Entropy).
    
    Args:
        z1, z2: Normalized embeddings from two augmented views
        temperature: Temperature scaling parameter
    """
    batch_size = z1.shape[0]
    
    # Concatenate embeddings
    z = torch.cat([z1, z2], dim=0)  # (2B, D)
    
    # Compute similarity matrix
    sim = torch.mm(z, z.t()) / temperature  # (2B, 2B)
    
    # Mask out self-similarities
    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim.masked_fill_(mask, float('-inf'))
    
    # Positive pairs: (i, i+B) and (i+B, i)
    pos_mask = torch.zeros(2 * batch_size, 2 * batch_size, device=z.device).bool()
    pos_mask[:batch_size, batch_size:] = torch.eye(batch_size, device=z.device).bool()
    pos_mask[batch_size:, :batch_size] = torch.eye(batch_size, device=z.device).bool()
    
    # InfoNCE loss
    exp_sim = torch.exp(sim)
    pos_sim = exp_sim[pos_mask].reshape(2 * batch_size, 1)
    neg_sim = exp_sim[~mask].reshape(2 * batch_size, -1).sum(dim=1, keepdim=True)
    
    loss = -torch.log(pos_sim / neg_sim).mean()
    return loss

In [ ]:
def pretrain_contrastive(encoder, data_loader, epochs=10, lr=1e-3, temperature=0.5):
    """
    Pretrain encoder with contrastive learning.
    """
    model = ContrastiveModel(encoder).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    augmenter = SequenceAugmenter()
    
    history = {"loss": []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        
        for X_batch, _ in data_loader:  # Labels not used in pretraining
            X_batch = X_batch.to(device)
            
            # Create two augmented views
            X1 = torch.stack([augmenter(x) for x in X_batch])
            X2 = torch.stack([augmenter(x) for x in X_batch])
            
            # Forward pass
            z1 = model(X1)
            z2 = model(X2)
            
            # Compute loss
            loss = nt_xent_loss(z1, z2, temperature)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(data_loader)
        history["loss"].append(avg_loss)
        
        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Contrastive Loss: {avg_loss:.4f}")
    
    return model.encoder, history

# Pretrain with contrastive learning
print("\nPretraining with contrastive learning...")
encoder = ConfigurableCNN(kernel_preset="medium", num_classes=3)
pretrained_encoder, cl_history = pretrain_contrastive(encoder, train_loader, epochs=10)

In [ ]:
# Fine-tune the pretrained encoder
print("\nFine-tuning pretrained encoder...")
ft_history = train_cnn(pretrained_encoder, train_loader, val_loader, epochs=10)

## 4. Deep Reinforcement Learning

RL-based approach that learns to classify sequences through reward signals.

In [ ]:
class SequenceClassificationEnv:
    """
    RL environment for sequence classification.
    
    State: Current sequence embedding
    Action: Predicted class
    Reward: +1 for correct, -1 for incorrect
    """
    
    def __init__(self, sequences, labels, encoder):
        self.sequences = sequences
        self.labels = labels
        self.encoder = encoder
        self.encoder.eval()
        
        self.current_idx = 0
        self.num_classes = len(torch.unique(labels))
    
    def reset(self):
        """Reset to a random sequence."""
        self.current_idx = torch.randint(0, len(self.sequences), (1,)).item()
        return self._get_state()
    
    def _get_state(self):
        """Get current state (sequence embedding)."""
        with torch.no_grad():
            seq = self.sequences[self.current_idx:self.current_idx+1].to(device)
            return self.encoder.get_features(seq).squeeze(0)
    
    def step(self, action):
        """Take action and get reward."""
        true_label = self.labels[self.current_idx].item()
        reward = 1.0 if action == true_label else -1.0
        done = True  # Episode ends after one prediction
        return self._get_state(), reward, done, {"correct": action == true_label}

In [ ]:
class PolicyNetwork(nn.Module):
    """Policy network for REINFORCE algorithm."""
    
    def __init__(self, state_dim: int, num_actions: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions),
        )
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return F.softmax(self.net(state), dim=-1)
    
    def select_action(self, state: torch.Tensor) -> tuple:
        """Sample action from policy."""
        probs = self.forward(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

In [ ]:
def train_reinforce(env, policy, episodes=1000, lr=1e-3, gamma=0.99):
    """
    Train policy with REINFORCE algorithm.
    """
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    
    history = {"rewards": [], "accuracy": []}
    running_reward = 0
    correct_count = 0
    
    for episode in range(episodes):
        state = env.reset()
        
        # Collect trajectory (single step for classification)
        action, log_prob = policy.select_action(state)
        _, reward, done, info = env.step(action)
        
        # Update running stats
        running_reward = 0.99 * running_reward + 0.01 * reward
        correct_count = 0.99 * correct_count + 0.01 * float(info["correct"])
        
        # Policy gradient update
        loss = -log_prob * reward
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (episode + 1) % 200 == 0:
            history["rewards"].append(running_reward)
            history["accuracy"].append(correct_count)
            print(f"Episode {episode+1}/{episodes} - "
                  f"Running Reward: {running_reward:.4f}, "
                  f"Accuracy: {correct_count:.4f}")
    
    return history

In [ ]:
# Create environment and policy
encoder_rl = ConfigurableCNN(kernel_preset="medium", num_classes=3).to(device)

env = SequenceClassificationEnv(X_train, y_train, encoder_rl)

# Get state dimension
with torch.no_grad():
    sample_state = env.reset()
    state_dim = sample_state.shape[0]

policy = PolicyNetwork(state_dim, num_actions=3).to(device)

# Train with REINFORCE
print("\nTraining with REINFORCE...")
rl_history = train_reinforce(env, policy, episodes=1000)

## 5. Comparison and Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CNN Training
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Configurable CNN Training")
axes[0].legend()

# Contrastive Learning
axes[1].plot(cl_history["loss"], label="Contrastive Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Contrastive Pretraining")
axes[1].legend()

# RL Training
if rl_history["accuracy"]:
    axes[2].plot(rl_history["accuracy"], label="Running Accuracy")
    axes[2].set_xlabel("Episode (x200)")
    axes[2].set_ylabel("Accuracy")
    axes[2].set_title("REINFORCE Training")
    axes[2].legend()

plt.tight_layout()
plt.savefig("training_comparison.png", dpi=150)
plt.show()

## 6. Summary

This notebook demonstrated three training approaches:

| Approach | Best For | Pros | Cons |
|----------|----------|------|------|
| **Configurable CNN** | Standard classification with labeled data | Simple, fast, interpretable kernel sizes | Requires labeled data |
| **Contrastive Learning** | Limited labeled data | Self-supervised pretraining, learns transferable features | Two-stage training |
| **Deep RL** | Custom reward functions, sequential decisions | Flexible rewards, exploration | Sample inefficient, unstable |

### Recommended Usage:

1. **Start with Configurable CNN** - Quick baseline with different kernel sizes
2. **Use Contrastive Learning** - When you have lots of unlabeled sequences
3. **Try Deep RL** - When you need custom reward functions or sequential processing